# RPL Diagnostics: Per-Sentence Excess-RPL Autocorrelation Analysis

This notebook demonstrates the **Within-Treebank Distribution Diagnostics** experiment, which computes per-sentence excess-RPL (Random Projective Linearization) autocorrelation for UD treebanks.

**What this does:**
- Defines core functions for computing dependency distance (DD) sequences, lag-1 autocorrelation, and random projective linearizations
- Demonstrates the RPL computation on synthetic example sentences
- Loads pre-computed results for 30 strategically selected UD treebanks (50 RPL permutations each, 307,688 sentences total)
- Produces distribution diagnostics, structural subtype analysis, and publication-quality visualizations

**Key finding:** All 30 treebanks show pervasive compensatory anti-correlation (all p<0.001), with pooled estimate of -0.194.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-Colab packages needed — all imports are pre-installed

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import math
import os
from collections import defaultdict

import numpy as np
import scipy.stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-c3cfa4-sequential-dependency-distance-anti-corr/main/experiment_iter3_rpl_diagnostics/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {sum(len(ds['examples']) for ds in data['datasets'])} examples across {len(data['datasets'])} datasets")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

Tunable parameters for the RPL computation demo on synthetic sentences.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────
# Number of random projective linearization permutations per sentence
N_PERMUTATIONS = 5  # Original: 50

# Random seed for reproducibility
RNG_SEED = 42

## Core Computation Functions

These functions implement the dependency distance (DD) sequence computation, lag-1 autocorrelation, and random projective linearization (RPL) baseline.

In [ ]:
def compute_dd_nopunct(head_array: list, deprel_array: list) -> list:
    """Compute DD sequence excluding punctuation tokens (deprel='punct').
    Returns list of DD values for non-punct, non-root tokens in linear order."""
    dd_seq = []
    for i in range(len(head_array)):
        if deprel_array[i] == "punct":
            continue
        h = head_array[i]
        if h == 0:
            continue  # skip root
        position = i + 1  # 1-indexed
        dd = abs(position - h)
        dd_seq.append(dd)
    return dd_seq


def r1_standard(x: list) -> float:
    """Standard lag-1 autocorrelation for sequence x."""
    n = len(x)
    if n < 4:
        return float('nan')
    mean_x = sum(x) / n
    numerator = sum((x[i] - mean_x) * (x[i + 1] - mean_x) for i in range(n - 1))
    denominator = sum((x[i] - mean_x) ** 2 for i in range(n))
    if denominator == 0:
        return 0.0
    return numerator / denominator


def r1_prime(x: list) -> float:
    """Bias-corrected lag-1 autocorrelation (Huitema & McKean 2000).
    r1' = r1 + (1 - r1^2) / (n - 3)"""
    n = len(x)
    if n < 5:
        return float('nan')
    r1 = r1_standard(x)
    if math.isnan(r1):
        return float('nan')
    correction = (1 - r1 ** 2) / (n - 3)
    return r1 + correction


def build_tree(head_array: list) -> tuple:
    """Build children lookup from head_array (1-indexed).
    Returns (dict: parent -> [children], root_position)."""
    children = defaultdict(list)
    root = None
    for i, h in enumerate(head_array):
        pos = i + 1  # 1-indexed position
        if h == 0:
            root = pos
        else:
            children[h].append(pos)
    return dict(children), root


def random_projective_linearization(children: dict, root: int, rng) -> list:
    """Generate one random projective linearization of the tree.
    Returns: list of original token positions in new linear order."""
    def linearize_subtree(node):
        kids = children.get(node, [])
        if not kids:
            return [node]
        child_subtrees = [linearize_subtree(c) for c in kids]
        perm = rng.permutation(len(child_subtrees)).tolist()
        shuffled = [child_subtrees[p] for p in perm]
        insert_pos = rng.integers(0, len(shuffled) + 1)
        result = []
        for idx, subtree in enumerate(shuffled):
            if idx == insert_pos:
                result.append(node)
            result.extend(subtree)
        if insert_pos == len(shuffled):
            result.append(node)
        return result

    if root is None:
        return []
    return linearize_subtree(root)


def dd_from_linearization(new_order: list, head_array: list, deprel_array: list) -> list:
    """Compute DD sequence from a linearization, excluding punct tokens."""
    new_pos = {}
    for new_idx, orig_pos in enumerate(new_order):
        new_pos[orig_pos] = new_idx + 1

    dd_seq = []
    for new_idx, orig_pos in enumerate(new_order):
        orig_idx = orig_pos - 1
        if orig_idx < 0 or orig_idx >= len(deprel_array):
            continue
        if deprel_array[orig_idx] == "punct":
            continue
        h = head_array[orig_idx]
        if h == 0:
            continue
        if h not in new_pos:
            continue
        dd = abs(new_pos[orig_pos] - new_pos[h])
        dd_seq.append(dd)
    return dd_seq


def compute_tree_depth(children: dict, root: int) -> int:
    """DFS to find max depth from root."""
    if root is None:
        return 0
    stack = [(root, 0)]
    max_depth = 0
    while stack:
        node, depth = stack.pop()
        max_depth = max(max_depth, depth)
        for child in children.get(node, []):
            stack.append((child, depth + 1))
    return max_depth


def compute_branching_factor(children: dict) -> float:
    """Mean number of children for nodes that have children."""
    branch_counts = [len(kids) for kids in children.values() if kids]
    return float(np.mean(branch_counts)) if branch_counts else 0.0


print("Core computation functions defined.")

## RPL Computation Demo on Synthetic Sentences

We demonstrate the core algorithm on two hand-crafted dependency trees, computing observed autocorrelation vs. the RPL baseline to get "excess-RPL".

In [ ]:
# Synthetic example sentences (head_array is 0-indexed head pointers, 0=root)
# Sentence 1: "The big cat sat on the warm mat ."  (9 tokens)
#   Tree: sat(4) -> The(1), big(2)->cat(3), on(5)->mat(8)->the(6),warm(7), .(9)
example_sentences = [
    {
        "sentence_id": "synth_001",
        "head_array": [3, 3, 4, 0, 4, 8, 8, 5, 4],
        "deprel_array": ["det", "amod", "nsubj", "root", "obl", "det", "amod", "nmod", "punct"],
    },
    {
        "sentence_id": "synth_002",
        "head_array": [2, 4, 2, 0, 4, 7, 4, 9, 7, 4],
        "deprel_array": ["det", "nsubj", "amod", "root", "nsubj", "det", "obj", "amod", "nmod", "punct"],
    },
]

rng = np.random.default_rng(RNG_SEED)

for sent in example_sentences:
    head_array = sent["head_array"]
    deprel_array = sent["deprel_array"]

    # Observed DD (excluding punct)
    dd_obs = compute_dd_nopunct(head_array, deprel_array)
    observed_r1 = r1_prime(dd_obs)

    # Build tree and generate RPL permutations
    children, root = build_tree(head_array)
    tree_depth = compute_tree_depth(children, root)
    bf = compute_branching_factor(children)

    rpl_r1_values = []
    for _ in range(N_PERMUTATIONS):
        new_order = random_projective_linearization(children, root, rng)
        dd_perm = dd_from_linearization(new_order, head_array, deprel_array)
        if len(dd_perm) >= 5:
            r1_val = r1_prime(dd_perm)
            if not math.isnan(r1_val):
                rpl_r1_values.append(r1_val)

    rpl_mean = float(np.mean(rpl_r1_values)) if rpl_r1_values else float('nan')
    excess = observed_r1 - rpl_mean if not math.isnan(rpl_mean) else float('nan')

    print(f"--- {sent['sentence_id']} ---")
    print(f"  Tokens: {len(head_array)}, DD length (no punct): {len(dd_obs)}")
    print(f"  Tree depth: {tree_depth}, Branching factor: {bf:.2f}")
    print(f"  DD sequence: {dd_obs}")
    print(f"  Observed r1': {observed_r1:.4f}")
    print(f"  RPL mean r1' ({len(rpl_r1_values)} valid perms): {rpl_mean:.4f}")
    print(f"  Excess-RPL: {excess:.4f}")
    print()

## Parse Pre-Computed Results

Extract the per-treebank diagnostics and per-sentence details from the loaded data for analysis and visualization.

In [ ]:
# Parse datasets into lookup dicts
datasets_map = {ds["dataset"]: ds["examples"] for ds in data["datasets"]}

# Parse treebank diagnostics
treebank_diagnostics = {}
for ex in datasets_map["treebank_distribution_diagnostics"]:
    tb_id = ex["metadata_treebank_id"]
    output = json.loads(ex["output"]) if isinstance(ex["output"], str) else ex["output"]
    treebank_diagnostics[tb_id] = output

# Parse aggregate summary
agg_output = json.loads(datasets_map["aggregate_summary"][0]["output"])

# Parse per-sentence details
sentence_details = []
for ex in datasets_map.get("per_sentence_details", []):
    output = json.loads(ex["output"]) if isinstance(ex["output"], str) else ex["output"]
    output["treebank_id"] = ex["metadata_treebank_id"]
    sentence_details.append(output)

# Parse structural subtype analysis
subtype_analysis = {}
for ex in datasets_map.get("structural_subtype_analysis", []):
    tb_id = ex["metadata_treebank_id"]
    output = json.loads(ex["output"]) if isinstance(ex["output"], str) else ex["output"]
    subtype_analysis[tb_id] = output

print(f"Parsed {len(treebank_diagnostics)} treebank diagnostics")
print(f"Parsed {len(sentence_details)} per-sentence details")
print(f"Parsed {len(subtype_analysis)} subtype analyses")
print(f"\nAggregate summary:")
print(f"  Treebanks: {agg_output['n_treebanks']}")
print(f"  All pervasive: {agg_output['n_pervasive']}/{agg_output['n_treebanks']}")
print(f"  All significant (p<0.001): {agg_output['n_significant_p001']}/{agg_output['n_treebanks']}")
print(f"  Pooled estimate: {agg_output['pooled_estimate']:.4f}")
print(f"  Spearman case vs excess: rho={agg_output['spearman_case_vs_excess']['rho']:.4f}, "
      f"p={agg_output['spearman_case_vs_excess']['p']:.4f}")

## Within-Treebank Distribution Diagnostics

Print a summary table of per-treebank results: proportion of negative excess-RPL, mean excess, significance, and pervasiveness classification.

In [ ]:
# Print diagnostics table
header = f"{'Treebank':>20s} {'Language':>12s} {'N':>7s} {'PropNeg':>8s} {'MeanExcess':>11s} {'p-value':>10s} {'Pervasive':>10s}"
print(header)
print("-" * len(header))

sorted_diag = sorted(treebank_diagnostics.items(), key=lambda x: x[1]["mean_excess_rpl"])
for tb_id, d in sorted_diag:
    p_str = f"{d['p_value']:.2e}" if d['p_value'] > 0 else "<1e-300"
    print(f"{tb_id:>20s} {str(d.get('language', '?')):>12s} {d['n_sentences']:>7d} "
          f"{d['prop_negative_excess']:>8.3f} {d['mean_excess_rpl']:>11.4f} "
          f"{p_str:>10s} {d['pervasiveness_class']:>10s}")

# Summary stats
all_means = [d["mean_excess_rpl"] for d in treebank_diagnostics.values()]
all_props = [d["prop_negative_excess"] for d in treebank_diagnostics.values()]
print(f"\nOverall: median mean_excess = {np.median(all_means):.4f}, "
      f"median prop_negative = {np.median(all_props):.3f}")
print(f"All {len(treebank_diagnostics)} treebanks classified as: "
      f"{', '.join(set(d['pervasiveness_class'] for d in treebank_diagnostics.values()))}")

## Visualization

Publication-quality figures reproducing the key results: forest plot of per-treebank estimates, pervasiveness barplot, word-order boxplot, and case-marking scatter.

In [ ]:
plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# ── FIGURE 1: Forest plot of per-treebank excess-RPL estimates ──────────
ax = axes[0, 0]
sorted_tbs = sorted(treebank_diagnostics.items(), key=lambda x: x[1]["mean_excess_rpl"])
for i, (tb_id, diag) in enumerate(sorted_tbs):
    mean_val = diag["mean_excess_rpl"]
    se = diag["se_mean_excess"]
    ci_lo = mean_val - 1.96 * se
    ci_hi = mean_val + 1.96 * se
    color = 'red' if diag.get("modality") == "spoken" else 'steelblue'
    ax.plot([ci_lo, ci_hi], [i, i], color=color, linewidth=1)
    ax.plot(mean_val, i, 'o', color=color, markersize=4)

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
# Pooled estimate
weights, weighted_vals = [], []
for _, d in sorted_tbs:
    if d["se_mean_excess"] > 0:
        w = 1 / d["se_mean_excess"] ** 2
        weights.append(w)
        weighted_vals.append(d["mean_excess_rpl"] * w)
pooled = sum(weighted_vals) / sum(weights) if weights else 0
ax.axvline(x=pooled, color='darkgreen', linestyle='-', linewidth=2, alpha=0.7,
           label=f'Pooled = {pooled:.4f}')
ax.set_yticks(list(range(len(sorted_tbs))))
ax.set_yticklabels([f"{tb_id} ({diag.get('language', '?')})" for tb_id, diag in sorted_tbs], fontsize=6)
ax.set_xlabel('Mean Excess-RPL (95% CI)')
ax.set_title('Forest Plot: Per-Treebank Excess-RPL')
ax.legend(loc='lower right', fontsize=8)

# ── FIGURE 2: Pervasiveness barplot ─────────────────────────────────────
ax = axes[0, 1]
sorted_tbs_perv = sorted(treebank_diagnostics.items(), key=lambda x: x[1]["prop_negative_excess"])
props_bar = [d["prop_negative_excess"] for _, d in sorted_tbs_perv]
colors_bar = ['darkred' if p > 0.6 else 'orange' if p > 0.5 else 'gray' for p in props_bar]
ax.barh(list(range(len(sorted_tbs_perv))), props_bar, color=colors_bar, alpha=0.8)
ax.axvline(x=0.5, color='gray', linestyle='--')
ax.axvline(x=0.6, color='red', linestyle='--', alpha=0.5, label='60% threshold')
ax.set_yticks(list(range(len(sorted_tbs_perv))))
ax.set_yticklabels([tb_id for tb_id, _ in sorted_tbs_perv], fontsize=6)
ax.set_xlabel('Proportion negative excess-RPL')
ax.set_title('Pervasiveness of Anti-Correlation')
ax.legend(fontsize=8)

# ── FIGURE 3: Boxplot by word order ─────────────────────────────────────
ax = axes[1, 0]
word_order_groups = defaultdict(list)
for tb_id, diag in treebank_diagnostics.items():
    wo = diag.get("word_order") or "Unknown"
    word_order_groups[wo].append(diag["mean_excess_rpl"])

order = [k for k in ["SOV", "SVO", "VSO", "VOS", "No dominant order", "Unknown"]
         if k in word_order_groups]
data_boxes = [word_order_groups[k] for k in order]
labels = [f"{k}\n(n={len(word_order_groups[k])})" for k in order]
colors_box = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9E9E9E', '#BDBDBD']
bp = ax.boxplot(data_boxes, tick_labels=labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_box[:len(order)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.7)
ax.set_ylabel('Mean excess-RPL')
ax.set_title('Excess-RPL by Word Order Type')

# ── FIGURE 4: Scatter — case-marking vs excess ─────────────────────────
ax = axes[1, 1]
case_vals_plot, excess_vals_plot = [], []
for tb_id, diag in treebank_diagnostics.items():
    x_val = diag.get("ud_case_proportion")
    if x_val is None:
        x_val = 0
    y_val = diag["mean_excess_rpl"]
    color = 'red' if diag.get("modality") == "spoken" else 'steelblue'
    marker = 's' if diag.get("modality") == "spoken" else 'o'
    ax.scatter(x_val, y_val, c=color, marker=marker, s=50, alpha=0.7,
               edgecolors='black', linewidths=0.5)
    if diag.get("ud_case_proportion") is not None:
        case_vals_plot.append(x_val)
        excess_vals_plot.append(y_val)

if len(case_vals_plot) >= 3:
    rho_val, p_rho_val = scipy.stats.spearmanr(case_vals_plot, excess_vals_plot)
    ax.annotate(f'Spearman rho = {rho_val:.3f}, p = {p_rho_val:.4f}',
                xy=(0.05, 0.95), xycoords='axes fraction', fontsize=9,
                ha='left', va='top')

ax.set_xlabel('UD Case Feature Proportion')
ax.set_ylabel('Mean Excess-RPL')
ax.set_title('Case-Marking vs. Anti-Correlation')
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', label='Written', markersize=8),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='red', label='Spoken', markersize=8),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=8)

plt.tight_layout()
plt.show()
print("All 4 figures generated successfully.")